In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

# Settings
N_SIMS = 100_000
SEED = 42
EXPLAINED_THRESHOLD = 0.99
DATA_DIR = Path.cwd() / "testfiles_" / "data"
CSV_PATH = DATA_DIR / "test5_2.csv"   # change if needed

# Read covariance (first row has column names)
df = pd.read_csv(CSV_PATH, header=0)
cols = list(df.columns)
Sigma = df.to_numpy(dtype=float)

# Symmetrize and make PSD (clip tiny negative eigenvalues)
Sigma = (Sigma + Sigma.T) / 2.0
eigvals, eigvecs = np.linalg.eigh(Sigma)
eigvals = np.clip(eigvals, 0.0, None)

# Sort eigenvalues/vectors descending
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

# Choose k s.t. cumulative explained variance >= 99%
total_var = eigvals.sum() + 1e-18
cum = np.cumsum(eigvals) / total_var
k = int(np.searchsorted(cum, EXPLAINED_THRESHOLD) + 1)

# PCA factor B_k = V_k * sqrt(D_k); target cov = V_k D_k V_k^T
sqrt_dk = np.sqrt(eigvals[:k])
B = eigvecs[:, :k] * sqrt_dk  # broadcasting columns

# Simulate X ~ N(0, V_k D_k V_k^T)
rng = np.random.default_rng(SEED)
Z = rng.standard_normal(size=(N_SIMS, k))
X = Z @ B.T

# Sample covariance (output only)
S_hat = np.cov(X, rowvar=False, ddof=1)

print(", ".join(cols))
for i in range(S_hat.shape[0]):
    row = ", ".join(f"{S_hat[i, j]:.16f}" for j in range(S_hat.shape[1]))
    print(row)


x1, x2, x3, x4, x5
0.0847974888285766, 0.1165315767991782, 0.0420944724179140, 0.0089659097146396, 0.0038654572097706
0.1165315767991782, 0.1601416336603417, 0.0578476475324111, 0.0123212563357646, 0.0053120420183070
0.0420944724179140, 0.0578476475324111, 0.0374870452008861, 0.0060142337119564, 0.0025808682617786
0.0089659097146396, 0.0123212563357646, 0.0060142337119564, 0.0010953271490882, 0.0004710921016116
0.0038654572097706, 0.0053120420183070, 0.0025808682617786, 0.0004710921016116, 0.0002026207865113
